In [ ]:
suppressMessages(library(scRepertoire))
suppressMessages(library(Seurat))
suppressMessages(library(tidyverse))

In [ ]:
getwd()

In [ ]:
library (reticulate)
library(anndata)
library(vegan)
library(dplyr)
library(ggsci)


library(circlize)
library(scales)

In [ ]:
seurat<- readRDS("TCR/pool_TCR/seurat_TCR.rds")

In [ ]:
project_ID="pool_TCR"
mtx = "VDJ"

In [ ]:
# List all directories under subdirectory
all_dirs <- list.dirs(paste0(mtx,"/"), full.names = TRUE, recursive = FALSE)


In [ ]:
contig_list <- list()
i <- 1

# Find all matrix.csv files in VDJ directories
matrix_files <- list.files(
  path = "VDJ", 
  pattern = "T.*matrix\\.csv$", , 
  recursive = TRUE, 
  full.names = TRUE
)

# Read each file
for (file in matrix_files) {
  contig_list[[i]] <- read.csv(file)
  i <- i + 1
}

In [ ]:
sample_list <- sapply(basename(matrix_files), function(x) {
  strsplit(x, "_")[[1]][1]
})

In [ ]:
combined <- combineTCR(contig_list
                     ,samples =sample_list)

In [ ]:
combined = combined[c("V40-T","V42-T","V21-T","V47-T"
                 )]

In [ ]:
combined <- addVariable(combined, variable.name = "treatment",
                        variables = c("oAd.1","oAd.2","oAd-mCD40L.1","oAd-mCD40L.2"))

In [ ]:
combined <- addVariable(combined, variable.name = "treatment2",
                        variables = c("oAd","oAd","oAd-mCD40L","oAd-mCD40L"))

In [ ]:
library(stringr)

# Get Seurat cell info
seurat_df <- data.frame(
  cell_name = colnames(seurat),
  barcode = str_extract(colnames(seurat), "^[^-]+"),
  sample = seurat$Sample_id
)

# Process each sample in the combined list separately
all_matches <- list()

for(sample_name in names(combined)) {
  # Clean sample name (remove -T)
  clean_sample <- str_replace(sample_name, "-T$", "")
  
  # Get TCR data for this sample
  tcr_data <- combined[[sample_name]]
  
  # Extract clean barcode from the barcode column (remove sample_ prefix)
  tcr_df <- data.frame(
    tcr_cell_name = tcr_data$barcode,
    barcode = str_extract(tcr_data$barcode, "[^_]+$")  # Gets everything after last "_"
  )
  
  # Get Seurat cells for this sample
  seurat_sample <- seurat_df[seurat_df$sample == clean_sample, ]
  
  # Find matches for this sample
  matches <- merge(seurat_sample, tcr_df, by = "barcode")
  
  if(nrow(matches) > 0) {
    all_matches[[sample_name]] <- matches
  }
}

# Combine all matches
final_matches <- do.call(rbind, all_matches)

# Create mapping and filter Seurat
cell_mapping <- final_matches$tcr_cell_name
names(cell_mapping) <- final_matches$cell_name

seurat<- subset(seurat, cells = names(cell_mapping))
seurat <- RenameCells(seurat, new.names = cell_mapping)

In [ ]:
Idents(seurat) <- seurat@meta.data$Sample_id

In [ ]:
contig_list_2  <- list()
i=1
for (samples in sample_list){
    old_barcodes <- combined[[samples]]$barcode
new_barcodes <- sapply(old_barcodes, function(x){

    cell <- sub("-1$","",x)

    cell


})
    
combined[[samples]]["barcode"]=new_barcodes
    
TCR <- combined[[samples]]
combined[[samples]] <-TCR[c( TCR$barcode%in%row.names(subset(seurat, idents = samples)@meta.data)),]
i=i+1
}



In [ ]:
cloneCall = "gene"
chain = "both"
scale = TRUE
exporTable = FALSE

In [ ]:
seurat <- combineExpression(combined, seurat, cloneCall = cloneCall,# group.by = "Sample_id",
                            proportion=FALSE,
                            cloneSize  = c(Single=1, Small=5, Medium=20, Large =100, Hyperexpanded =500))

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",chain,".pdf"))
clonalAbundance(combined, cloneCall = cloneCall, chain = chain, scale=scale)
dev.off()

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",chain,"_coverage.pdf"), width=5, height=3, pointsize=0.1)
clonalRarefaction(combined,
                  plot.type = 2,
                  hill.numbers = 1,
                  n.boots = 2)
dev.off()

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",chain,"_bar.pdf"))
clonalQuant(combined, cloneCall = cloneCall, chain = chain,scale=scale, group.by="treatment")
dev.off()

## individual treatment

In [ ]:
seurat@meta.data$site_treatment = paste0(seurat@meta.data$site, "_", seurat@meta.data$treatment)

In [ ]:
seurat@meta.data$treat_anno = paste0(seurat@meta.data$anno_sub, "_", seurat@meta.data$treatment)

In [ ]:
seurat@meta.data$site_treat_key = paste0(seurat@meta.data$anno_key, "_", seurat@meta.data$site_treatment)

In [ ]:
seurat@meta.data$site_anno = paste0( seurat@meta.data$site,"_", seurat@meta.data$anno_sub2)

In [ ]:
seurat@meta.data$anno_key = sub("^([^.]+\\.[^.]+).*","\\1",seurat@meta.data$anno_sub2)

In [ ]:
Idents(seurat) <- seurat@meta.data$site_anno

In [ ]:
treatment_g="M20"

In [ ]:
seurat_sub<- subset(seurat, subset = treatment == treatment_g)

In [ ]:
cloneCall = "gene+nt"
chain = "TRA"
scale = TRUE
exporTable = FALSE

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",treatment_g,"_",cloneCall,"_",chain,"_overlap.pdf"), width=12, height=10, pointsize=5)
library(viridis)

p <- clonalOverlap(seurat_sub,cloneCall = cloneCall, method = "jaccard",group.by="treat_anno", palette = "Plasma", 
                  #  palette = "Batlow",
                   exportTable=F)


p+theme(axis.text.x = element_text(#face="bold", color="#993333", 
                           size=10, angle=45),
          axis.text.y = element_text(#face="bold", color="#993333", 
                           size=10, angle=0))
p
dev.off()

In [ ]:
overlap_<-clonalOverlap(seurat_sub,cloneCall = cloneCall, method = "jaccard",group.by="treat_anno", palette = "Plasma", 
                  #  palette = "Batlow",
                   exportTable=T)

In [ ]:
Idents(combined_sub) <- combined_sub@meta.data$loc_anno_sub

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",treatment_g,"_",cloneCall,"_",chain,"_network.pdf"), width=20, height=6, pointsize=0.6)

for (start in unique(seurat_sub@meta.data$treatment_anno)){
    #start = unique(combined_sub@meta.data$loc_anno_sub)[20]


# Save as variable to global environment
cells_to_highlight <- WhichCells(seurat_sub, idents=start)

p1<-DimPlot(seurat_sub, reduction="umap",cells.highlight=cells_to_highlight,cols.highlight = "red" , cols="grey" )

p2 <- clonalNetwork(seurat_sub, reduction="umap",group.by="treatment_anno",
             filter.identity=start ,filter.proportion=0.1,
              cloneCall=cloneCall, exportTable=FALSE, palette = "Roma" )+ 
  scale_color_manual(values = c(paletteer_d("ggsci::default_ucscgb") ,paletteer_d("ggsci::default_igv"))[1:length(unique(combined_sub@meta.data$loc_anno_sub))])
    
p3 <- DimPlot(seurat_sub, reduction="umap",group.by="treatment_anno"  )+ 
  scale_color_manual(values = c(paletteer_d("ggsci::default_ucscgb") ,paletteer_d("ggsci::default_igv"))[1:length(unique(combined_sub@meta.data$loc_anno_sub))])
    
p <- p1+p2+p3

plot(p,main=paste0(treatment, "_", cloneCall,"_",chain,"_",start))

    }
dev.off()

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",chain,"_cloneType.pdf"), width=18, height=6, pointsize=0.6)
DimPlot(seurat, group.by="cloneSize",split.by="treatment")
dev.off()

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",chain,"_cloneOverlay.pdf"), width=18, height=6, pointsize=0.6)
clonalOverlay(seurat, reduction="umap", bins = 12, facet ="treatment")+guides(color="none")
dev.off()

In [ ]:
require(scales)
library(ggplot2)

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",chain,"_Occupy2.pdf"), width=30, height=6, pointsize=0.6)
clonalOccupy(
 seurat,
  x.axis = "site_treat_anno",
  label = TRUE,
  facet.by = NULL,
  order.by = NULL,
  proportion = FALSE,
  na.include = FALSE,
  exportTable = FALSE,
  palette = "inferno")+
  theme(axis.text.x = element_text(angle = 90))
dev.off()

In [ ]:
# Run clustering, but group calculations by "Patient"
seurat <- clonalCluster( seurat, 
                               chain = "TRA", 
                               sequence = "aa", 
                               threshold = 0.85, 
                               group.by = "site_treatment")


In [ ]:
#Define color palette 
num_clusters <- length(unique(na.omit(seurat$TRA_cluster)))
cluster_colors <- hcl.colors(n = num_clusters, palette = "inferno")

DimPlot(seurat, group.by = "TRA_cluster") +
  scale_color_manual(values = cluster_colors, na.value = "grey") + 
  NoLegend()

In [ ]:
summary_table <- seurat@meta.data %>%
  group_by(site_treat_anno) %>%
  summarize(
    n_unique_TRA_clusters = n_distinct(TRA_cluster, na.rm = TRUE),
    total_cells = n()
  ) %>%
  arrange(desc(n_unique_TRA_clusters))

write.csv(summary_table,paste0(mtx,"/",project_ID,"/",cloneCall,"_",chain,"_TRA_cluster_diversity.csv"))

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",chain,"_diversity.pdf"), width=12, height=3, pointsize=0.2)
clonalDiversity(seurat,cloneCall = cloneCall, group.by="site_treat_anno")
dev.off()

In [ ]:
#all TCR
meta <- seurat@meta.data
diversity_method="shannon"
# 用 TRA_cluster 作为 clone 列
meta <- meta %>%
  mutate(
    clone_adj = CTstrict  # 保持原始克隆类型，不进行"other"合并
  )

shannon_table <- meta %>%
  group_by(site_treatment, anno_sub2) %>%
  summarise(
    Clone_Shannon = diversity(table(clone_adj), index = diversity_method),
    .groups = "drop"
  ) %>%
  mutate(Shannon = round(Clone_Shannon, 4))
shannon_table$site_treat_anno=paste0(shannon_table$anno_sub2,"_",shannon_table$site_treatment)


In [ ]:
write.csv(shannon_table,paste0(mtx,"/",project_ID,"/",cloneCall,"_",chain,"_cloneDiv",diversity_method,".csv"))

In [ ]:
seurat@meta.data$treat_anno=paste0(seurat@meta.data$anno_sub2,"_",seurat@meta.data$treatment)

In [ ]:
# 统计每个样本 cluster 的多样性
cluster_div <- seurat@meta.data %>%
    group_by(treat_anno) %>%
    summarise(Cluster_Shannon = {
        tab <- table(TRA_cluster)
        diversity(tab,index=diversity_method)
    })


In [ ]:
div_df <- left_join(shannon_table, cluster_div, by = "treat_anno")


In [ ]:
div_df$Clone_Shannon<- ifelse (div_df$Clone_Shannon > 50,50,div_df$Clone_Shannon)
div_df$Cluster_Shannon<- ifelse (div_df$Cluster_Shannon > 50,50,div_df$Cluster_Shannon)
div_df<-div_df %>%
  mutate(across(where(is.numeric), ~ ifelse(is.infinite(.), 50, .)))


In [ ]:
# Create barplots with all site_treatments in one plot
pdf(paste0(mtx, "/", project_ID, "/", cloneCall, "_", chain, "_cloneDiv_barplots.pdf"), 
    width = 12, height = 8, pointsize = 0.4)

# Define the color palette
color_palette <- c(
  "#FF7F00", "#FFA500", "#FF8C00",  # Oranges
  "#00A087", "#20B2AA", "#40E0D0", "#48D1CC", "#00CED1", "#5F9EA0", "#008080",  # Teals
  "#E64B35", "#DC0000", "#C72C48", "#FF6B6B",  # Reds
  "#E64B35", "#D94E8F", "#C71585", "#DA70D6", "#BA55D3", "#9370DB",  # Pinks/Purples
  "#808080"  # Gray
)

# Plot 1: All site_treatments in one barplot (Clone Diversity)
p1 <- ggplot(div_df, aes(x = site_treatment, y = Clone_Shannon, fill = anno_sub2)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.8), alpha = 0.8) +
  scale_fill_manual(values = color_palette[1:length(unique(div_df$anno_sub2))]) +
  theme_classic(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Clone Diversity - All Site-Treatments",
       x = "Site-Treatment",
       y = paste0("Clone Diversity (",diversity_method," index)"),
       fill = "Cell Type")

print(p1)

# Plot 2: All site_treatments in one barplot (Cluster Diversity)
p2 <- ggplot(div_df, aes(x = site_treatment, y = Cluster_Shannon, fill = anno_sub2)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.8), alpha = 0.8) +
  scale_fill_manual(values = color_palette[1:length(unique(div_df$anno_sub2))]) +
  theme_classic(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Cluster Diversity - All Site-Treatments",
       x = "Site-Treatment",
       y = paste0("Cluster Diversity (",diversity_method," index)"),
       fill = "Cell Type")

print(p2)

# Individual barplots for each cell type
anno_categories <- unique(div_df$anno_sub2)

for (i in seq_along(anno_categories)) {
  anno_cat <- anno_categories[i]
  
  # Filter data for current cell type
  filtered_df <- div_df[div_df$anno_sub2 == anno_cat, ]
  
  # Create barplot for this cell type (Clone Diversity)
  p_clone <- ggplot(filtered_df, aes(x = site_treatment, y = Clone_Shannon, fill = site_treatment)) +
    geom_bar(stat = "identity", alpha = 0.8) +
    scale_fill_manual(values = color_palette[1:length(unique(filtered_df$site_treatment))]) +
    theme_classic(base_size = 12) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
    labs(title = paste("Clone Diversity -", anno_cat),
         x = "Site-Treatment",
         y = paste0("Clone Diversity (",diversity_method," index)"),
         fill = "Site-Treatment")
  
  # Create barplot for this cell type (Cluster Diversity)
  p_cluster <- ggplot(filtered_df, aes(x = site_treatment, y = Cluster_Shannon, fill = site_treatment)) +
    geom_bar(stat = "identity", alpha = 0.8) +
    scale_fill_manual(values = color_palette[1:length(unique(filtered_df$site_treatment))]) +
    theme_classic(base_size = 12) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
    labs(title = paste("Cluster Diversity -", anno_cat),
         x = "Site-Treatment",
         y = paste0("Cluster Diversity (",diversity_method," index)"),
         fill = "Site-Treatment")
  
  print(p_clone)
  print(p_cluster)
  
  cat("Barplots created for cell type:", anno_cat, "\n")
}

dev.off()

cat("All barplots saved to PDF file\n")

## every two

In [ ]:
group_names <-unique(seurat@meta.data$site_anno)

# The numeric indices in circles correspond to positions in group_names
# So index 1 = group_names[1], index 2 = group_names[2], etc.

# Create color mapping using the correct order
color_list_45 <- c(
  "#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FECA57",
  "#FF9FF3", "#54A0FF", "#5F27CD", "#00D2D3", "#FF9F43",
  "#EE5A24", "#C4E538", "#12CBC4", "#FDA7DF", "#ED4C67",
  "#A3CB38", "#1289A7", "#D980FA", "#B53471", "#FFC312",
  "#C4E538", "#12CBC4", "#FDA7DF", "#ED4C67", "#A3CB38",
  "#1289A7", "#D980FA", "#B53471", "#FFC312", "#EE5A24",
  "#5758BB", "#6F1E51", "#1B1464", "#006266", "#EA2027",
  "#FF3838", "#32FF7E", "#7BED9F", "#67E6DC", "#17C0EB",
  "#7158E2", "#3D3D3D", "#AAAAAA", "#FF9F1A", "#2ED573"
)
# Assign colors based on the order of group_names
master_color_map <- setNames(color_list_45[1:length(group_names)], group_names)


In [ ]:
treatment_g="IM.1"
cloneCall="CTgene"

In [ ]:
seurat_sub<- subset(seurat, subset = treatment == treatment_g)

In [ ]:

Idents(seurat_sub) <- seurat_sub@meta.data$site_anno
circles <- getCirclize(seurat_sub, group.by = "site_anno", proportion = TRUE, include.self=FALSE)
circles[is.na(circles)] <- 0
present_groups <- levels(seurat_sub@active.ident)
grid.cols <- master_color_map[present_groups]

#grid.cols <- scales::hue_pal()(length(unique(seurat_sub@active.ident)))
#names(grid.cols) <- levels(seurat_sub@active.ident)
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",treatment_g,"_",chain,"_clone_compare_all.pdf"), width=12, height=12, pointsize=0.4)
chordDiagram(circles, 
             self.link = 0, 
             grid.col = grid.cols, 
             directional = 1, 
             link.target.prop = TRUE,
             direction.type =  "arrows",
             link.arr.type = "big.arrow",
             
             # Font size and label customization
             annotationTrack = "grid",
             preAllocateTracks = list(
               track.height = 0.4,  # Adjust height for labels
               track.margin = c(0.02, 0.02)  # Adjust margins
             ))

# Customize the labels after drawing the diagram
circos.track(track.index = 1, panel.fun = function(x, y) {
  circos.text(CELL_META$xcenter, 
              CELL_META$ylim[1], 
              CELL_META$sector.index, 
              facing = "clockwise",
              niceFacing = TRUE,
              adj = c(0, 0.5),
              cex = 1.8,  # Font size here
              col = "black",
              font = 2)   # Bold font
}, bg.border = NA)
dev.off()

In [ ]:
alluvialClones(seurat, 
               cloneCall = "aa", 
               
               y.axes = c("anno_sub", "site"), 
            color = "ident"  )

In [ ]:
seurat_sub_TDLN<- subset(seurat_sub, subset = site == "TDLN")
seurat_sub_Tumor<- subset(seurat_sub, subset = site == "Tumor")

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",treatment_g,"_",chain,"_clone_compare_TDLN2Tumor.pdf"), width=12, height=12, pointsize=0.4)
for (anno1 in unique(seurat_sub_TDLN@meta.data$site_treat_anno)){
    for (anno2 in unique(seurat_sub_Tumor@meta.data$site_treat_anno)){
        if (anno1 != anno2){
            
            # Subset the data
            seurat_sub_ = subset(seurat_sub, subset = site_treat_anno %in% c(anno1, anno2))
            Idents(seurat_sub_) <- seurat_sub_@meta.data$site_treat_anno
            
            # Check if both groups have cells
            cell_counts <- table(seurat_sub_@meta.data$site_treat_anno)
            if(any(cell_counts == 0)) {
                message(paste("Skipping", anno1, "vs", anno2, "- one group has no cells"))
                next
            }
            
            # Determine safe number of top clones
            total_clones <- length(unique(seurat_sub_@meta.data[[cloneCall]]))
            top_n <- min(20, total_clones)
            
            if(top_n > 0) {
                p <- clonalCompare(seurat_sub_, 
                          top.clones = top_n, 
                          samples = c(anno1, anno2),  # Fixed: removed str()
                          relabel.clones = TRUE,
                          cloneCall = cloneCall, 
                          graph = "alluvial")
                
                plot(p + ggtitle(paste("Clonal Comparison:", anno1, "vs", anno2)) +
                         theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold")))
            } else {
                message(paste("No clones found for", anno1, "vs", anno2))
            }
        }
    }
}
dev.off()

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",treatment_g,"_",chain,"_clone_compare_TDLN2TDLN.pdf"), width=12, height=12, pointsize=0.4)
for (anno1 in unique(seurat_sub_TDLN@meta.data$site_treat_anno)){
    for (anno2 in unique(seurat_sub_TDLN@meta.data$site_treat_anno)){
        if (anno1 != anno2){
            
            # Subset the data
            seurat_sub_ = subset(seurat_sub_TDLN, subset = site_treat_anno %in% c(anno1, anno2))
            Idents(seurat_sub_) <- seurat_sub_@meta.data$site_treat_anno
            
            # Check if both groups have cells
            cell_counts <- table(seurat_sub_@meta.data$site_treat_anno)
            if(any(cell_counts == 0)) {
                message(paste("Skipping", anno1, "vs", anno2, "- one group has no cells"))
                next
            }
            
            # Determine safe number of top clones
            total_clones <- length(unique(seurat_sub_@meta.data[[cloneCall]]))
            top_n <- min(20, total_clones)
            
            if(top_n > 0) {
                p <- clonalCompare(seurat_sub_, 
                          top.clones = top_n, 
                          samples = c(anno1, anno2),  # Fixed: removed str()
                          relabel.clones = TRUE,
                          cloneCall = cloneCall, 
                          graph = "alluvial")
                
                plot(p + ggtitle(paste("Clonal Comparison:", anno1, "vs", anno2)) +
                         theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold")))
            } else {
                message(paste("No clones found for", anno1, "vs", anno2))
            }
        }
    }
}
dev.off()

In [ ]:
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",treatment_g,"_",chain,"_clone_compare_Tumor2Tumor.pdf"), width=12, height=12, pointsize=0.4)
for (anno1 in unique(seurat_sub_Tumor@meta.data$site_treat_anno)){
    for (anno2 in unique(seurat_sub_Tumor@meta.data$site_treat_anno)){
        if (anno1 != anno2){
            
            # Subset the data
            seurat_sub_ = subset(seurat_sub_Tumor, subset = site_treat_anno %in% c(anno1, anno2))
            Idents(seurat_sub_) <- seurat_sub_@meta.data$site_treat_anno
            
            # Check if both groups have cells
            cell_counts <- table(seurat_sub_@meta.data$site_treat_anno)
            if(any(cell_counts == 0)) {
                message(paste("Skipping", anno1, "vs", anno2, "- one group has no cells"))
                next
            }
            
            # Determine safe number of top clones
            total_clones <- length(unique(seurat_sub_@meta.data[[cloneCall]]))
            top_n <- min(20, total_clones)
            
            if(top_n > 0) {
                p <- clonalCompare(seurat_sub_, 
                          top.clones = top_n, 
                          samples = c(anno1, anno2),  # Fixed: removed str()
                          relabel.clones = TRUE,
                          cloneCall = cloneCall, 
                          graph = "alluvial")
                
                plot(p + ggtitle(paste("Clonal Comparison:", anno1, "vs", anno2)) +
                         theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold")))
            } else {
                message(paste("No clones found for", anno1, "vs", anno2))
            }
        }
    }
}
dev.off()

In [ ]:
pdf(paste0(outputdir, treatment, "_", cloneCall,"_",chain,"_network.pdf"), width=20, height=6, pointsize=0.6)

for (start in unique(combined_sub@meta.data$loc_anno_sub)){
    #start = unique(combined_sub@meta.data$loc_anno_sub)[20]


# Save as variable to global environment
cells_to_highlight <- WhichCells(combined_sub, idents=start)

p1<-DimPlot(combined_sub, reduction="umap",cells.highlight=cells_to_highlight,cols.highlight = "red" , cols="grey" )

p2 <- clonalNetwork(combined_sub, reduction="umap",group.by="loc_anno_sub",
             filter.identity=start ,filter.proportion=0.1,
              cloneCall=cloneCall, exportTable=FALSE, palette = "Roma" )+ 
  scale_color_manual(values =  c(paletteer_d("ggsci::default_ucscgb") ,paletteer_d("ggsci::default_igv"))[c(1:(length(unique(combined_sub@meta.data$loc_anno_sub))-2),20,21)])
    
p3 <- DimPlot(combined_sub, reduction="umap",group.by="loc_anno_sub"  )+ 
  scale_color_manual(values =  c(paletteer_d("ggsci::default_ucscgb") ,paletteer_d("ggsci::default_igv"))[c(1:(length(unique(combined_sub@meta.data$loc_anno_sub))-2),20,21)])
    
p <- p1+p2+p3

plot(p,main=paste0(treatment, "_", cloneCall,"_",chain,"_",start))

    }
dev.off()

In [ ]:
groupby = "anno_sub"
#pdf(paste0(outputdir,treatment, "_",groupby,"_UMAP.pdf"),
#    width=5, height=3, pointsize=0.2)

DimPlot(combined, group.by = groupby) +
    scale_color_manual(values = c(
                                  paletteer_d("ggsci::default_igv"))[1:length(unique(combined_sub@meta.data[groupby])[[1]])])
#dev.off()

In [ ]:
X_axis="V47"
Y_axis="V4"
pdf(paste0(mtx,"/",project_ID,"/",cloneCall,"_",chain,"_",X_axis,"_",Y_axis,"_corr.pdf"), width=4, height=3, pointsize=0.2)
clonalScatter(combined, cloneCall= cloneCall, x.axis = X_axis, y.axis = Y_axis)
dev.off()

In [ ]:
set.seed(42)
#Clustering Patient 19 samples
igraph.object <- clonalCluster(combined[[]],
                               chain = "TRA",
                               sequence = "aa",
                               group.by = "sample",
                               threshold = 0.85, 
                               exportGraph = TRUE)

# Setting color scheme
col_legend <- factor(igraph::V(igraph.object)$group)
col_samples <- hcl.colors(2,"inferno")[as.numeric(col_legend)]
color.legend <- factor(unique(igraph::V(igraph.object)$group))

# Sampling 1000 Barcodes
sample.vertices <- V(igraph.object)[sample(length(igraph.object), 1000)]
subgraph.object <- induced_subgraph(igraph.object, vids = sample.vertices)
V(subgraph.object)$degrees <- igraph::degree(subgraph.object)
edge_alpha_color <- adjustcolor("gray", alpha.f = 0.3)

#Plotting
plot(subgraph.object,
     layout = layout_nicely(subgraph.object),
     vertex.label = NA,
     vertex.size = sqrt(igraph::V(subgraph.object)$degrees), 
     vertex.color = col_samples[sample.vertices],
     vertex.frame.color = "white", 
     edge.color = edge_alpha_color,
     edge.arrow.size = 0.05,
     edge.curved = 0.05, 
     margin = -0.1)
legend("topleft", 
       legend = levels(color.legend), 
       pch = 16, 
       col = unique(col_samples), 
       bty = "n")

In [ ]:
library(ggraph)

In [ ]:
exportClones(combined,
            format="paired",
            group.by = "site_treatment",
            write.file=T,
            dir = NULL,
            file.name="clone.tmp.csv")

In [ ]:
saveRDS(seurat, paste0(mtx,"/",project_ID,"/","seurat_TCR.rds"))